# Upload a PDF to Deka Box (Cloudeka's S3-compatible storage)

**Deka Box** is Cloudeka's object storage. It speaks the **S3 API**, so we can use the standard AWS SDK for Python, **`boto3`**, and just point it at the Deka Box endpoint instead of AWS.

In this notebook we upload a **PDF document** to a bucket:

1. Configure credentials and the Deka Box endpoint.
2. Build an S3 client for Deka Box.
3. Upload a PDF into a bucket.
4. Verify it landed, and generate a link to it.

Later RAG notebooks read source PDFs from a bucket like this one.

## Dependencies

This notebook needs `boto3` (the S3 client) and `python-dotenv` (loads credentials from the shared `var.env` file so they stay out of the notebook). They are listed in `../requirements.txt`; install once from the `day2skk` folder:

```bash
pip install -r requirements.txt
```

## Configuration

All notebooks in `day2skk` share **one** environment file: **`day2skk/var.env`** (one level up from this notebook), loaded with `load_dotenv("../var.env")`. You don't create a per-notebook `.env` - just add your values to that shared file.

Get your **Access Key**, **Secret Key**, and **endpoint** from the Deka Box console (S3 credentials / access keys section), then add these lines to `../var.env`:

```dotenv
ACCESS_KEY_ID=your-access-key
SECRET_ACCESS_KEY=your-secret-key
S3_ENDPOINT_URL=https://your-deka-box-endpoint   # from the Deka Box console
REGION=us-east-1                                 # any value works for most S3-compatible stores
S3_BUCKET=my-bucket                              # an existing bucket you can write to
```

Never commit real keys - keep `var.env` in your `.gitignore`.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv("../var.env")   # shared env file at day2skk/var.env

ACCESS_KEY_ID = os.environ.get("ACCESS_KEY_ID")
SECRET_ACCESS_KEY = os.environ.get("SECRET_ACCESS_KEY")
S3_ENDPOINT_URL = os.environ.get("S3_ENDPOINT_URL")
REGION = os.environ.get("REGION", "us-east-1")
S3_BUCKET = os.environ.get("S3_BUCKET")

# Sanity check without printing the secrets themselves.
print("ACCESS_KEY_ID set:    ", bool(ACCESS_KEY_ID))
print("SECRET_ACCESS_KEY set:", bool(SECRET_ACCESS_KEY))
print("S3_ENDPOINT_URL:      ", S3_ENDPOINT_URL)
print("REGION:               ", REGION)
print("S3_BUCKET:            ", S3_BUCKET)

ACCESS_KEY_ID set:     True
SECRET_ACCESS_KEY set: True
S3_ENDPOINT_URL:       https://kencana.basic.box.cloudeka.id
REGION:                kencana
S3_BUCKET:             skk


## Build the Deka Box client

This is a normal `boto3` S3 client with two Deka-specific touches:

- **`endpoint_url`** points at Deka Box instead of AWS.
- **The checksum `Config`**: recent `botocore` (>= 1.36) adds a default CRC32 checksum that forces `aws-chunked` / `Transfer-Encoding: chunked`, which some S3-compatible providers (Cloudeka/Ceph/MinIO) reject with `NotImplemented`. Setting the checksum behaviour to `when_required` avoids that.

In [2]:
import boto3
from botocore.config import Config


def make_client():
    """Build an S3 client pointed at Deka Box."""
    return boto3.client(
        "s3",
        region_name=REGION,
        endpoint_url=S3_ENDPOINT_URL or None,
        aws_access_key_id=ACCESS_KEY_ID,
        aws_secret_access_key=SECRET_ACCESS_KEY,
        config=Config(
            request_checksum_calculation="when_required",
            response_checksum_validation="when_required",
        ),
    )


client = make_client()
print("client ready for endpoint:", client.meta.endpoint_url)

client ready for endpoint: https://kencana.basic.box.cloudeka.id


## Upload the PDF

We use **`put_object`** with the file read as **bytes**, rather than the higher-level `upload_file`. `upload_file` streams with `Transfer-Encoding: chunked`, which some S3-compatible providers reject. Sending bytes gives a known `Content-Length` and avoids chunked encoding - the reliable choice for Deka Box.

We also pass **`ContentType="application/pdf"`** so Deka Box stores and serves the object with the correct type (e.g. a browser opening the presigned link renders the PDF instead of downloading it as unknown data).

In [3]:
from botocore.exceptions import BotoCoreError, ClientError
from pathlib import Path

def upload(client, bucket: str, file_path: Path, key: str,
           content_type: str = "application/pdf") -> None:
    print(f"Uploading {file_path} -> s3://{bucket}/{key}")
    with open(file_path, "rb") as f:
        client.put_object(Bucket=bucket, Key=key, Body=f.read(), ContentType=content_type)
    print("  upload OK")

file_path = Path("cloudeka.pdf")
object_key = f"uploads/{file_path.name}"

if not S3_BUCKET:
    raise ValueError("S3_BUCKET is not set. Add it to the shared ../var.env file.")

try:
    upload(client, S3_BUCKET, Path("cloudeka.pdf"), object_key)
except (BotoCoreError, ClientError) as exc:
    print("S3 error:", exc)

Uploading cloudeka.pdf -> s3://skk/uploads/cloudeka.pdf
  upload OK


## Verify the upload

`head_object` fetches just the metadata of the object (size, last-modified) to confirm it exists, and `list_objects_v2` shows what is under our `uploads/` prefix.

In [4]:
head = client.head_object(Bucket=S3_BUCKET, Key=object_key)
print("exists:", object_key)
print("  size (bytes):", head["ContentLength"])
print("  last modified:", head["LastModified"])

print("\nObjects under 'uploads/':")
listing = client.list_objects_v2(Bucket=S3_BUCKET, Prefix="uploads/")
for obj in listing.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

exists: uploads/cloudeka.pdf
  size (bytes): 1471988
  last modified: 2026-07-25 23:32:04+00:00

Objects under 'uploads/':
  uploads/cloudeka.pdf  (1471988 bytes)
  uploads/sample.pdf  (355302 bytes)


## Recap

- **Deka Box is S3-compatible**, so `boto3` works once you set `endpoint_url` to the Deka Box endpoint and pass your access/secret keys.
- Keep credentials in the shared **`var.env`** file (`day2skk/var.env`), not in the notebook.
- Use the **checksum `Config`** (`when_required`) so newer `botocore` does not force chunked encoding that Cloudeka can reject.
- Upload with **`put_object(Body=bytes, ContentType="application/pdf")`** rather than `upload_file`, to avoid chunked `Transfer-Encoding` and tag the object as a PDF.
- Verify with **`head_object`** / **`list_objects_v2`**, and share with a **presigned URL**.

The same pattern uploads the source PDFs that the RAG pipeline will ingest.